# Notebook 00b — Verification 

Loads clean data from Delta, generates plots to verify:
1. Overview — all runs on one canvas
2. Per-run cycle plots (feed · COT · dd_abs_max · pass-level)
3. Suspicious run deep-dives (warmup/tail anomalies)
4. dd_abs_max trajectory overlay (all runs, x = run_age_days)
5. Run quality summary (bar charts)

All plots saved to `output/1HA/00b_clean/verify/`


## 1. Setup

In [0]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 10,
})

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, REPO_ROOT)

os.environ['DDF_CLUSTER_ID'] = '0612-154044-62gqpkte'

from olefins_ddf.io_events import get_spark

FURNACE       = '1HA'
DELTA_CATALOG = 'indorama_corporate_olefins_paas_azure_weu_dev_research.workspaces'
OUTPUT_DIR    = os.path.join(REPO_ROOT, 'output', FURNACE, '00b_clean', 'verify')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Thresholds
ALARM_C     = 45.0
PREALARM_C  = 30.0
PASS_COLORS = {'A':'#1f77b4','B':'#ff7f0e','C':'#2ca02c','D':'#d62728'}

print(f'Output: {OUTPUT_DIR}')


## 2. Load data

In [0]:
spark = get_spark()

# Full feature matrix (all rows including warmup/tail/decoke)
feat = spark.table(f'{DELTA_CATALOG}.{FURNACE.lower()}_features').toPandas()
ts_col = next(c for c in ['timestamp','ts'] if c in feat.columns)
feat = feat.set_index(ts_col)
feat.index = pd.to_datetime(feat.index)
feat = feat.sort_index()
feat.index.name = 'timestamp'

# Clean data (analysis windows only, has run_id)
clean = spark.table(f'{DELTA_CATALOG}.{FURNACE.lower()}_clean').toPandas()
ts_col = next(c for c in ['timestamp','ts'] if c in clean.columns)
clean = clean.set_index(ts_col)
clean.index = pd.to_datetime(clean.index)
clean = clean.sort_index()
clean.index.name = 'timestamp'

# Run windows CSV (metadata)
win_path = os.path.join(REPO_ROOT, 'output', FURNACE, '00b_clean', 'run_analysis_windows.csv')
win_df = pd.read_csv(win_path, parse_dates=['run_start','run_end','analysis_start','analysis_end'])

run_ids = sorted(clean['run_id'].unique())
print(f'Feature matrix : {feat.shape[0]:,} rows')
print(f'Clean data     : {clean.shape[0]:,} rows')
print(f'Complete runs  : {len(run_ids)}')
display(win_df[['run','run_start','run_end','warmup_hours','clean_days','tail_hours','length_days']])


## 3. Per-run cycle plots

Full cycle (decoke→decoke) with green=clean, red=warmup/tail. 4 panels each.

In [0]:
PASS_COLS = {p: f'dd_abs_max_{p}' for p in 'ABCD' if f'dd_abs_max_{p}' in feat.columns}
cot_col = next((c for c in ['cot','cot_ctrl'] if c in feat.columns), None)

def plot_one_cycle(run_id, feat_full, clean_df, win_df, out_dir):
    w = win_df[win_df['run'] == run_id]
    if w.empty: return
    w = w.iloc[0]
    run_start      = pd.Timestamp(w['run_start'])
    run_end        = pd.Timestamp(w['run_end'])
    analysis_start = pd.Timestamp(w['analysis_start'])
    analysis_end   = pd.Timestamp(w['analysis_end'])
    clean_days     = (analysis_end - analysis_start).total_seconds()/86400
    total_days     = (run_end - run_start).total_seconds()/86400

    seg   = feat_full.loc[run_start:run_end]
    seg_c = clean_df[clean_df['run_id'] == run_id]
    if seg.empty: return

    n = 4 if PASS_COLS else 3
    fig, axes = plt.subplots(n, 1, figsize=(14, 3.0*n), sharex=True)
    fig.suptitle(
        f'{FURNACE}  |  Run {run_id}  |  '
        f'{run_start.date()} → {run_end.date()}  '
        f'({total_days:.1f} d total · {clean_days:.1f} d clean · '
        f'warmup {w["warmup_hours"]:.0f}h · tail {w["tail_hours"]:.0f}h)',
        fontsize=10, fontweight='bold'
    )

    def shade(ax):
        ax.axvspan(run_start,      analysis_start, alpha=0.13, color='#e24b4a', zorder=0, label='warmup/tail')
        ax.axvspan(analysis_start, analysis_end,   alpha=0.10, color='#1d9e75', zorder=0, label='clean')
        ax.axvspan(analysis_end,   run_end,        alpha=0.13, color='#e24b4a', zorder=0)
        # Flag if anomalous
        if w['warmup_hours'] > 48:
            ax.axvspan(run_start, analysis_start, alpha=0.08, color='#7f77dd', zorder=0)
        if w['tail_hours'] > 24:
            ax.axvspan(analysis_end, run_end, alpha=0.08, color='#7f77dd', zorder=0)

    # Feed
    ax = axes[0]
    if 'feed_total' in seg.columns:
        ax.plot(seg.index,   seg['feed_total'],   color='#378add', lw=0.7, alpha=0.6, label='raw')
        ax.plot(seg_c.index, seg_c['feed_total'], color='#1d9e75', lw=1.2, label='clean')
    shade(ax); ax.set_ylabel('Feed (NM³/H)'); ax.legend(fontsize=7, loc='upper right')

    # COT
    ax = axes[1]
    if cot_col:
        ax.plot(seg.index,   seg[cot_col],   color='#7f77dd', lw=0.7, alpha=0.6, label='raw')
        ax.plot(seg_c.index, seg_c[cot_col], color='#085041', lw=1.2, label='clean')
    shade(ax); ax.set_ylabel('COT (°C)'); ax.legend(fontsize=7, loc='upper right')

    # dd_abs_max
    ax = axes[2]
    if 'dd_abs_max' in seg.columns:
        ax.plot(seg.index,   seg['dd_abs_max'],   color='#ef9f27', lw=0.6, alpha=0.5, label='raw')
        ax.plot(seg_c.index, seg_c['dd_abs_max'], color='#d85a30', lw=1.2, label='clean')
        ax.axhline(ALARM_C,    color='#e24b4a', ls='--', lw=1.0, label=f'Alarm {ALARM_C}°C')
        ax.axhline(PREALARM_C, color='#ef9f27', ls='--', lw=1.0, label=f'Pre-alarm {PREALARM_C}°C')
    shade(ax); ax.set_ylabel('dd_abs_max (°C)'); ax.legend(fontsize=7, loc='upper right')

    # Pass-level
    if PASS_COLS and n == 4:
        ax = axes[3]
        for p, col in PASS_COLS.items():
            c = PASS_COLORS[p]
            ax.plot(seg.index,   seg[col],   lw=0.5, alpha=0.35, color=c)
            ax.plot(seg_c.index, seg_c[col], lw=1.0, color=c, label=f'Pass {p}')
        ax.axhline(ALARM_C,    color='#e24b4a', ls='--', lw=0.8)
        ax.axhline(PREALARM_C, color='#ef9f27', ls='--', lw=0.8)
        shade(ax); ax.set_ylabel('dd per pass (°C)'); ax.legend(fontsize=7, loc='upper right', ncol=2)

    # Annotate anomalies
    if w['warmup_hours'] > 48:
        axes[0].text(run_start + (analysis_start - run_start)/2, axes[0].get_ylim()[1]*0.95,
                     f'⚠ warmup\n{w["warmup_hours"]:.0f}h', ha='center', va='top',
                     fontsize=8, color='#7f77dd', fontweight='bold')
    if w['tail_hours'] > 24:
        axes[0].text(analysis_end + (run_end - analysis_end)/2, axes[0].get_ylim()[1]*0.95,
                     f'⚠ tail\n{w["tail_hours"]:.0f}h', ha='center', va='top',
                     fontsize=8, color='#7f77dd', fontweight='bold')

    axes[-1].set_xlabel('Date')
    plt.tight_layout()
    path = os.path.join(out_dir, f'{FURNACE}_run{run_id:02d}_cycle.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Saved: run {run_id:02d} → {path}')

print(f'Generating {len(run_ids)} per-cycle plots...')
for rid in run_ids:
    plot_one_cycle(rid, feat, clean, win_df, OUTPUT_DIR)
print('Done.')


## 4. dd_abs_max trajectory overlay

All runs on same x-axis (run_age_days). Shows coking rate across runs.

In [0]:
if 'dd_abs_max' in clean.columns:
    fig, ax = plt.subplots(figsize=(13, 5))
    cmap = plt.cm.viridis
    colors = [cmap(i/max(len(run_ids)-1,1)) for i in range(len(run_ids))]

    for idx, rid in enumerate(run_ids):
        w = win_df[win_df['run'] == rid].iloc[0]
        seg = clean[clean['run_id'] == rid][['dd_abs_max']].copy()
        if seg.empty: continue
        seg['run_age_days'] = (seg.index - w['analysis_start']).total_seconds()/86400
        seg = seg[seg['run_age_days'] >= 0]
        ax.plot(seg['run_age_days'], seg['dd_abs_max'],
                lw=1.2, alpha=0.75, color=colors[idx], label=f'Run {rid} ({str(w["run_start"])[:10]})')

    ax.axhline(ALARM_C,    color='#e24b4a', ls='--', lw=1.2, label=f'Alarm {ALARM_C}°C')
    ax.axhline(PREALARM_C, color='#ef9f27', ls='--', lw=1.2, label=f'Pre-alarm {PREALARM_C}°C')
    ax.set_xlabel('Run age (days from analysis_start)')
    ax.set_ylabel('dd_abs_max (°C)')
    ax.set_title(f'{FURNACE} — dd_abs_max trajectories, all {len(run_ids)} runs (colour = chronological order)')
    ax.legend(fontsize=7, loc='upper left', ncol=3, framealpha=0.7)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f'{FURNACE}_dd_trajectories_overlay.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Saved: {path}')


## 5. Run quality summary — bar charts

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

r_ids    = win_df['run'].tolist()
warmups  = win_df['warmup_hours'].tolist()
tails    = win_df['tail_hours'].tolist()
cleans   = win_df['clean_days'].tolist()
lengths  = win_df['length_days'].tolist()

# Colors: flag anomalies
def warmup_color(h): return '#e24b4a' if h>48 else ('#ef9f27' if h>30 else '#1d9e75')
def tail_color(h):   return '#e24b4a' if h>48 else ('#ef9f27' if h>24 else '#1d9e75')

# Plot 1: Warmup
ax = axes[0]
bars = ax.bar(r_ids, warmups, color=[warmup_color(h) for h in warmups], edgecolor='white', lw=0.5)
ax.axhline(48, color='#e24b4a', ls='--', lw=1.0, label='48h threshold')
ax.axhline(12, color='#1d9e75', ls=':', lw=0.8, label='Typical 12h')
ax.set_xlabel('Run ID'); ax.set_ylabel('Warmup (hours)')
ax.set_title('Warmup duration per run\n(red = needs DCS verification)')
ax.legend(fontsize=8); ax.set_xticks(r_ids)
for bar, val in zip(bars, warmups):
    if val > 48: ax.text(bar.get_x()+bar.get_width()/2, val+2, f'{val:.0f}h', ha='center', va='bottom', fontsize=7, fontweight='bold', color='#e24b4a')

# Plot 2: Tail
ax = axes[1]
bars = ax.bar(r_ids, tails, color=[tail_color(h) for h in tails], edgecolor='white', lw=0.5)
ax.axhline(24, color='#ef9f27', ls='--', lw=1.0, label='24h threshold')
ax.set_xlabel('Run ID'); ax.set_ylabel('Tail (hours)')
ax.set_title('Tail duration per run\n(red = needs DCS verification)')
ax.legend(fontsize=8); ax.set_xticks(r_ids)
for bar, val in zip(bars, tails):
    if val > 24: ax.text(bar.get_x()+bar.get_width()/2, val+2, f'{val:.0f}h', ha='center', va='bottom', fontsize=7, fontweight='bold', color='#e24b4a')

# Plot 3: Clean vs total
ax = axes[2]
bar_l = ax.bar(r_ids, lengths, color='#eeedea', edgecolor='var(--border)', lw=0.5, label='Total run')
bar_c = ax.bar(r_ids, cleans,  color='#1d9e75', alpha=0.8, lw=0, label='Clean window')
ax.axhline(np.mean(cleans), color='#085041', ls='--', lw=1.0, label=f'Avg clean {np.mean(cleans):.1f}d')
ax.set_xlabel('Run ID'); ax.set_ylabel('Days')
ax.set_title('Total run vs clean window\n(green = kept for analysis)')
ax.legend(fontsize=8); ax.set_xticks(r_ids)

plt.suptitle(f'{FURNACE} — Run quality summary ({len(r_ids)} complete cycles)', fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, f'{FURNACE}_run_quality_summary.png')
plt.savefig(path, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved: {path}')


## 6. Suspicious run deep-dives

Zoomed detail for runs with anomalous warmup/tail.

In [0]:
suspicious = win_df[(win_df['warmup_hours'] > 48) | (win_df['tail_hours'] > 24)].copy()
print(f'Suspicious runs: {suspicious["run"].tolist()}')

for _, w in suspicious.iterrows():
    rid = int(w['run'])
    run_start = pd.Timestamp(w['run_start'])
    run_end   = pd.Timestamp(w['run_end'])
    a_start   = pd.Timestamp(w['analysis_start'])
    a_end     = pd.Timestamp(w['analysis_end'])

    # Add 12h context before and after
    ctx_start = run_start - pd.Timedelta(hours=12)
    ctx_end   = run_end   + pd.Timedelta(hours=12)
    seg = feat.loc[ctx_start:ctx_end]

    fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
    issue = f'warmup {w["warmup_hours"]:.0f}h' if w['warmup_hours']>48 else f'tail {w["tail_hours"]:.0f}h'
    fig.suptitle(f'{FURNACE}  |  Run {rid}  |  {run_start.date()} → {run_end.date()}  |  ⚠ {issue} — verify with DCS',
                 fontsize=10, fontweight='bold', color='#a32d2d')

    def shade_detail(ax):
        ax.axvspan(ctx_start, run_start, alpha=0.08, color='#aaaaaa', zorder=0)
        ax.axvline(run_start, color='#aaaaaa', lw=1.0, ls='--')
        ax.axvspan(run_start, a_start,   alpha=0.15, color='#e24b4a', zorder=0, label='warmup')
        ax.axvspan(a_start,   a_end,     alpha=0.10, color='#1d9e75', zorder=0, label='clean')
        ax.axvspan(a_end,     run_end,   alpha=0.15, color='#ef9f27', zorder=0, label='tail')
        ax.axvline(run_end,   color='#aaaaaa', lw=1.0, ls='--')
        ax.axvspan(run_end,   ctx_end,   alpha=0.08, color='#aaaaaa', zorder=0)

    if 'feed_total' in seg.columns:
        axes[0].plot(seg.index, seg['feed_total'], color='#378add', lw=0.8)
        axes[0].set_ylabel('Feed (NM³/H)')
    shade_detail(axes[0]); axes[0].legend(fontsize=7, loc='upper right')

    if cot_col and cot_col in seg.columns:
        axes[1].plot(seg.index, seg[cot_col], color='#7f77dd', lw=0.8)
        axes[1].set_ylabel('COT (°C)')
    shade_detail(axes[1])

    if 'dd_abs_max' in seg.columns:
        axes[2].plot(seg.index, seg['dd_abs_max'], color='#d85a30', lw=0.8)
        axes[2].axhline(ALARM_C,    color='#e24b4a', ls='--', lw=1.0)
        axes[2].axhline(PREALARM_C, color='#ef9f27', ls='--', lw=1.0)
        axes[2].set_ylabel('dd_abs_max (°C)')
    shade_detail(axes[2])

    # Annotate the anomalous zone
    if w['warmup_hours'] > 48:
        mid = run_start + (a_start - run_start)/2
        axes[0].annotate(f'⚠ Warmup {w["warmup_hours"]:.0f}h\n(expected ≤24h)\nVerify with DCS',
                         xy=(mid, axes[0].get_ylim()[1]*0.5), ha='center', fontsize=8,
                         color='#a32d2d', fontweight='bold',
                         bbox=dict(boxstyle='round,pad=0.3', fc='#fcebeb', ec='#e24b4a', alpha=0.9))
    if w['tail_hours'] > 24:
        mid = a_end + (run_end - a_end)/2
        axes[0].annotate(f'⚠ Tail {w["tail_hours"]:.0f}h\n(expected ≤6h)\nVerify with DCS',
                         xy=(mid, axes[0].get_ylim()[1]*0.5), ha='center', fontsize=8,
                         color='#a32d2d', fontweight='bold',
                         bbox=dict(boxstyle='round,pad=0.3', fc='#fcebeb', ec='#e24b4a', alpha=0.9))

    axes[-1].set_xlabel('Date')
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f'{FURNACE}_run{rid:02d}_deepdive.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Deep-dive saved: run {rid} → {path}')


## 7. Summary

In [0]:
import glob
pngs = glob.glob(os.path.join(OUTPUT_DIR, '*.png'))
print(f'All verification plots saved to: {OUTPUT_DIR}')
print(f'Total files: {len(pngs)}')
print()
for p in sorted(pngs):
    print(f'  {os.path.basename(p)}')
print()
print('Plots for Harry:')
print('  1. *_cycle.png      — one per run, full cycle view')
print('  2. *_deepdive.png   — suspicious runs zoomed in with annotations')
print('  3. *_trajectories_overlay.png — coking rate across all runs')
print('  4. *_run_quality_summary.png  — warmup/tail/clean bar charts')
